# Discovery + Silver: `billing.customers`

Primera tabla del dominio `billing`. Tiene un FK **opcional** hacia `university.students` via `external_ref` -- ya sabemos por `docs/calidad_datos.md` que esta vacio en el 50% de las filas a proposito (no todo cliente es estudiante), asi que lo tratamos como nullable, no como error.

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.billing__customers", engine)
df.shape

(10000, 11)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

customer_id             object
external_ref            object
first_name              object
last_name               object
email                   object
country                 object
created_at              object
segment                 object
_source_file            object
_ingested_at    datetime64[ns]
_dag_run_id             object
dtype: object


,customer_id,external_ref,first_name,last_name,email,country,created_at,segment,_source_file,_ingested_at,_dag_run_id
0,CUS-0000001,STU-0000001,Carlos,Contreras,carlos.contreras7090@example.com,CL,2021-07-14 19:53:33,smb,billing/customers.csv,2026-07-21 15:02:02.950371,manual__2026-07-21T15:01:58.988673+00:00
1,CUS-0000002,STU-0000002,Maria,Sandoval,maria.sandoval1480@lake.local,MX,2019-02-01 06:07:23,retail,billing/customers.csv,2026-07-21 15:02:02.950371,manual__2026-07-21T15:01:58.988673+00:00
2,CUS-0000003,STU-0000003,Ignacio,Torres,ignacio.torres8864@example.com,CL,2018-06-03 21:02:27,retail,billing/customers.csv,2026-07-21 15:02:02.950371,manual__2026-07-21T15:01:58.988673+00:00
3,CUS-0000004,STU-0000004,Juan,Sandoval,juan.sandoval3659@demo.io,MX,2019-08-29 12:43:31,smb,billing/customers.csv,2026-07-21 15:02:02.950371,manual__2026-07-21T15:01:58.988673+00:00
4,CUS-0000005,STU-0000005,Cristobal,Reyes,cristobal.reyes1471@mail.test,CL,2020-08-06 11:27:12,retail,billing/customers.csv,2026-07-21 15:02:02.950371,manual__2026-07-21T15:01:58.988673+00:00


## 2. Nulos y duplicados

`external_ref` nulo es esperado (~50%). El resto de columnas no deberia tener nulos.

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("customer_id duplicados:", df["customer_id"].duplicated().sum())
print("% external_ref nulo:", round(df["external_ref"].isna().mean() * 100, 1))

Nulos por columna:
customer_id        0
external_ref    5000
first_name         0
last_name          0
email              0
country            0
created_at         0
segment            0
_source_file       0
_ingested_at       0
_dag_run_id        0
dtype: int64

customer_id duplicados: 0
% external_ref nulo: 50.0


## 3. Integridad referencial de `external_ref` (FK opcional a `students`)

In [4]:
students = pd.read_sql("SELECT student_id FROM silver.university__students", engine)

con_ref = df[df["external_ref"].notna()]
huerfanos = ~con_ref["external_ref"].isin(students["student_id"])
print("Filas con external_ref no nulo:", len(con_ref))
print("external_ref huerfanos (no existe ese student_id):", huerfanos.sum())

Filas con external_ref no nulo: 5000
external_ref huerfanos (no existe ese student_id): 0


## 4. `country`, `segment`, `created_at`

In [5]:
print("country:")
print(df["country"].value_counts())
print()
print("segment:")
print(df["segment"].value_counts())
print()
created = pd.to_datetime(df["created_at"])
print("created_at invalidas:", created.isna().sum())

country:
country
CL    4053
PE     991
AR     989
MX     969
BR     817
ES     790
CO     775
US     616
Name: count, dtype: int64

segment:
segment
retail        7019
smb           2200
enterprise     781
Name: count, dtype: int64

created_at invalidas: 0


## 5. Conclusion: reglas de limpieza

Tabla limpia estructuralmente (sin nulos salvo `external_ref` intencional, sin duplicados, 0 FKs huerfanas). Reglas:

- `first_name`, `last_name` -> `strip()`.
- `email` -> `strip()` + minusculas.
- `country` -> `strip()` + mayusculas.
- `segment` -> `strip()` + minusculas.
- `created_at` -> castear a timestamp real (trae hora, no solo fecha).
- `external_ref` -> se mantiene nullable tal cual, **no se completa ni se descarta** (es un dato faltante legitimo, no un error).

## 6. Limpieza con pandas

In [6]:
df_silver = df[["customer_id", "external_ref", "first_name", "last_name", "email", "country", "created_at", "segment"]].copy()

df_silver["first_name"] = df_silver["first_name"].str.strip()
df_silver["last_name"] = df_silver["last_name"].str.strip()
df_silver["email"] = df_silver["email"].str.strip().str.lower()
df_silver["country"] = df_silver["country"].str.strip().str.upper()
df_silver["segment"] = df_silver["segment"].str.strip().str.lower()
df_silver["created_at"] = pd.to_datetime(df_silver["created_at"])

df_silver.head()

,customer_id,external_ref,first_name,last_name,email,country,created_at,segment
0,CUS-0000001,STU-0000001,Carlos,Contreras,carlos.contreras7090@example.com,CL,2021-07-14 19:53:33,smb
1,CUS-0000002,STU-0000002,Maria,Sandoval,maria.sandoval1480@lake.local,MX,2019-02-01 06:07:23,retail
2,CUS-0000003,STU-0000003,Ignacio,Torres,ignacio.torres8864@example.com,CL,2018-06-03 21:02:27,retail
3,CUS-0000004,STU-0000004,Juan,Sandoval,juan.sandoval3659@demo.io,MX,2019-08-29 12:43:31,smb
4,CUS-0000005,STU-0000005,Cristobal,Reyes,cristobal.reyes1471@mail.test,CL,2020-08-06 11:27:12,retail


## 7. Validar antes de escribir

In [7]:
assert len(df_silver) == len(df)
assert df_silver["customer_id"].is_unique
assert df_silver["external_ref"].dropna().isin(students["student_id"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 10000 filas listas para silver


## 8. Escribir en `silver.billing__customers`

In [8]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "billing.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.billing__customers CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "billing__customers",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=2000,
)
print("Escrito en silver.billing__customers")

OK: billing.sql ejecutado


Escrito en silver.billing__customers


## 9. Verificar

In [9]:
check = pd.read_sql("SELECT * FROM silver.billing__customers LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(DISTINCT customer_id) AS ids_unicos, count(external_ref) AS con_external_ref FROM silver.billing__customers", engine))
check

   filas  ids_unicos  con_external_ref
0  10000       10000              5000


,customer_id,external_ref,first_name,last_name,email,country,created_at,segment,_silver_loaded_at
0,CUS-0000001,STU-0000001,Carlos,Contreras,carlos.contreras7090@example.com,CL,2021-07-14 19:53:33,smb,2026-07-21 15:02:27.445995+00:00
1,CUS-0000002,STU-0000002,Maria,Sandoval,maria.sandoval1480@lake.local,MX,2019-02-01 06:07:23,retail,2026-07-21 15:02:27.445995+00:00
2,CUS-0000003,STU-0000003,Ignacio,Torres,ignacio.torres8864@example.com,CL,2018-06-03 21:02:27,retail,2026-07-21 15:02:27.445995+00:00
3,CUS-0000004,STU-0000004,Juan,Sandoval,juan.sandoval3659@demo.io,MX,2019-08-29 12:43:31,smb,2026-07-21 15:02:27.445995+00:00
4,CUS-0000005,STU-0000005,Cristobal,Reyes,cristobal.reyes1471@mail.test,CL,2020-08-06 11:27:12,retail,2026-07-21 15:02:27.445995+00:00
